# DATA ENGENERING PROJECT

NAME: SHAHAD BANDAR AL SHAMMARI

· [BRIEF.md](https://github.com/SDAIAAcademy/qafila-margin-project/blob/main/BRIEF.md). What I did in this project:
- landing the app database and every supplier price file in DuckDB, **exactly as they arrive**;
- cleaning them with dbt: costs parsed (`9,49`, `٩٫٤٩`), SKUs trimmed, Riyadh order dates;
- an **SCD2 history** of supplier costs, and a **quarantine** for rows we can't use;
- each order line costed at the price **valid on its order date**, rolled up to a weekly margin;
- publishing to **Iceberg** in place, re-running safely, and time travel after the supplier's correction.

Every code file of the project is written by a `%%writefile` cell


**Steps**


- [ ] 0 · Setup and data
- [ ] 1 · Land: `run.py` (`ingest`, `transform`, `publish`)
- [ ] 2 · Clean: dbt staging models
- [ ] 3 · History: quarantine and SCD2 cost history
- [ ] 4 · Margin: point-in-time costing and the weekly table
- [ ] 5 · Prove it: tests, a run, a re-run
- [ ] 6 · The correction arrives: update in place, time travel

## 0 · Setup and data

I used google colab so this clones the project template and installs the same packages as `pyproject.toml` (about a minute).
Locally, open the notebook from the project folder with the project's `.venv` as the kernel.

In [1]:
import os, sys, subprocess
from pathlib import Path



IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO_URL = "https://github.com/SDAIAAcademy/qafila-margin-project.git"
    if not Path("/content/qafila-margin-project").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/qafila-margin-project"], check=True)
    os.chdir("/content/qafila-margin-project")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "polars", "pyarrow", "pandas",
                    "dbt-core", "dbt-duckdb", "pyiceberg[sql-sqlite,pyarrow]"], check=True)
sys.path.insert(0, os.getcwd())
print("project:", os.getcwd(), "| colab:", IN_COLAB)


def py(*args):
    """Run one of the project's scripts, like `uv run python ...` in a terminal, and return its output as text."""
    r = subprocess.run([sys.executable, *args], capture_output=True, text=True, encoding="utf-8",
                       env={**os.environ, "PYTHONIOENCODING": "utf-8"})
    if r.returncode:
        print(r.stderr[-3000:])
    return r.stdout

project: C:\Users\vmuil\qafila-nbtest | colab: False


In [2]:
import shutil

import duckdb
import pandas as pd



pd.set_option("display.max_colwidth", 60)
shutil.rmtree("warehouse", ignore_errors=True)          # start from an empty warehouse, like the grader
print(py("tools/make_data.py"))                         # data/ (seed 2026, ≈35 MB)
for f in sorted(Path("data/suppliers").glob("*.csv")):
    print(f.name, "|", f.read_text(encoding="utf-8-sig").splitlines()[0])

wrote C:\Users\vmuil\qafila-nbtest\data
  customers 7,442 (snapshot 6,060, 6,346 CDC events)
  products 356 | stores 12 | orders 45,766 | order_items 172,545
  payments 41,785 | delivery events 22,518 | reviews 3,000

prices_2026-W36.csv | supplier_id,sku,product_name_ar,unit_cost,valid_from,min_order_qty
prices_2026-W37.csv | supplier_id,sku,product_name_ar,unit_cost,valid_from,min_order_qty
prices_2026-W38.csv | supplier_id,sku,product_name_ar,unit_cost,valid_from,min_order_qty
prices_2026-W39.csv | supplier_id,sku,product_name_ar,valid_from,min_order_qty,cost_sar,currency


Four weekly files. Week 39 renamed `unit_cost` to `cost_sar` and added `currency`. Week 38 also starts with a
byte-order mark (that's why it's read with `utf-8-sig` above). lets Look at a few raw values:

In [3]:
con = duckdb.connect()
con.sql("SELECT * FROM read_csv('data/suppliers/prices_2026-W38.csv', all_varchar = true) LIMIT 5").df()

,supplier_id,sku,product_name_ar,unit_cost,valid_from,min_order_qty
0,SUP-NAJD-DAIRY,QF-DAI-1047,زبدة الواحة,"9,49",2026-09-14,12
1,SUP-FRESHFARMS,QF-FRU-1109,بصل الواحة,"2,51",2026-09-14,6
2,SUP-FRESHFARMS,QF-FRU-1128,مانجو قافلة بيسك,"11,09",2026-09-14,48
3,SUP-HOMECARE,QF-HOU-1267,مناديل الواحة صغير,"5,59",2026-09-14,24
4,SUP-FRESHFARMS,QF-MEA-1147,صدور دجاج بركة,"18,63",2026-09-14,6


## 1 · Land: `run.py`

`run.py` is the one command: `ingest()` → `transform()` (dbt) → `publish()` (Iceberg). The template gave us
`get_catalog()`, `transform()` and `main()`; we write `ingest()` and `publish()`. The file is written in pieces below.

**Header, imports and constants.** `PUBLISH` lists the three tables and their key columns: the key is also the sort
order, so the same data always gives the same rows in the same order, and a re-run can be compared row by row.

In [4]:
%%writefile run.py
"""Qafila gross margin pipeline: raw files -> DuckDB -> dbt -> Iceberg.

    uv run python run.py --data-dir data --warehouse warehouse

Must work on an empty warehouse folder, and must be safe to run again on the same one.
The grader runs it on a dataset you haven't seen, so never hard-code file names, row counts or SKUs.
"""

from __future__ import annotations

import argparse
import os
import sqlite3
import subprocess
import sys
from contextlib import closing
from pathlib import Path

import duckdb
import polars as pl
from pyiceberg.catalog.sql import SqlCatalog

DBT_DIR = Path(__file__).resolve().parent / "dbt"
APP_TABLES = ["orders", "order_items", "products", "stores"]      # the four tables of the app database we need

# The dbt models published to Iceberg, with the columns that make one row unique (also the sort order we write in,
# so the same data always produces the same file and a re-run can be compared row by row).
PUBLISH = {
    "silver.product_cost_history": ["sku", "valid_from"],
    "gold.weekly_margin": ["iso_week", "city", "category"],
    "quality.price_quarantine": ["source_file", "sku", "valid_from", "raw_cost"],
}


def get_catalog(warehouse: Path) -> SqlCatalog:
    """The Iceberg catalog the grader reads. Don't change the name, the URI or the warehouse."""
    warehouse.mkdir(parents=True, exist_ok=True)
    return SqlCatalog("qafila", uri=f"sqlite:///{(warehouse / 'catalog.db').as_posix()}", warehouse=warehouse.resolve().as_posix())

Overwriting run.py


**`ingest()`**: both sources land in schema `raw`, untouched. Every supplier column is read as **text**
(`all_varchar`), so `'9,49'` and `'٩٫٤٩'` arrive exactly as written; deciding what they mean is staging's job.
`CREATE OR REPLACE` makes each run a full refresh: running twice never duplicates a row.

In [5]:
%%writefile -a run.py


def ingest(data_dir: Path, db: Path) -> None:
    """Step 1: land the sources in DuckDB, schema `raw`, exactly as they arrive.

    - From data_dir/qafila_oltp.sqlite: orders, order_items, products, stores.
    - Every data_dir/suppliers/prices_*.csv, keeping the file name of each row.

    Running this twice must not duplicate anything.
    """
    supplier_files = (data_dir / "suppliers" / "prices_*.csv").as_posix()   # DuckDB likes forward slashes, also on Windows

    with duckdb.connect(str(db)) as con:  # closes the file before dbt opens it
        con.sql("CREATE SCHEMA IF NOT EXISTS raw")

        # App database: read the four tables with Python's own sqlite3 driver and Polars (Lab 0 · Polars intro),
        # then copy them into raw as they are. DuckDB's ATTACH ... (TYPE sqlite) would need an extension that is
        # downloaded on first use, and the grader's machine may have no internet: this way needs nothing extra.
        # CREATE OR REPLACE = a full refresh, so running twice never duplicates a row.
        with closing(sqlite3.connect(data_dir / "qafila_oltp.sqlite")) as app:
            for table in APP_TABLES:
                # infer_schema_length=None: look at every row before choosing a type. promo_code is empty on
                # most orders, and guessing from the first 100 rows could type it as "always null".
                incoming = pl.read_database(f"SELECT * FROM {table}", app, infer_schema_length=None)
                con.register("incoming", incoming)
                con.sql(f"CREATE OR REPLACE TABLE raw.{table} AS SELECT * FROM incoming")
                con.unregister("incoming")

        # Supplier files: every column read as text (all_varchar), so nothing is "fixed" on the way in:
        # '9,49', '٩٫٤٩' and '-5.5' land exactly as the supplier wrote them, and staging decides what they mean.
        # union_by_name lines up the files by column name: W39 renamed unit_cost -> cost_sar and added currency.
        # filename = true keeps the file each row came from (rule 5 needs it to pick the winning file).
        con.sql(f"""
          CREATE OR REPLACE TABLE raw.supplier_prices AS
          SELECT * EXCLUDE (filename),
                 parse_filename(filename) AS source_file
          FROM read_csv('{supplier_files}', union_by_name = true, all_varchar = true, filename = true)
        """)
        # A dataset where no file uses one of the two cost columns would leave it out of the table,
        # and staging reads both: make sure both always exist (empty if no file has them).
        for col in ["unit_cost", "cost_sar"]:
            con.sql(f"ALTER TABLE raw.supplier_prices ADD COLUMN IF NOT EXISTS {col} VARCHAR")

        counts = {t: con.sql(f"SELECT count(*) FROM raw.{t}").fetchone()[0] for t in APP_TABLES + ["supplier_prices"]}
        n_files = con.sql("SELECT count(DISTINCT source_file) FROM raw.supplier_prices").fetchone()[0]
    print("ingest:", ", ".join(f"{t} {n:,}" for t, n in counts.items()), f"({n_files} supplier files)")

Appending to run.py

**`transform()`**: unchanged from the template. dbt runs in its own process, so the DuckDB file is free for `publish()` afterwards.

In [6]:
%%writefile -a run.py



def transform(db: Path) -> None:
    """Steps 2-4: build and test the dbt project in dbt/ against the DuckDB file.

    dbt runs in its own process: in this one it would keep the DuckDB file open and block publish().
    """
    env = {**os.environ, "QAFILA_DUCKDB": str(db.resolve())}
    cmd = [sys.executable, "-c", "from dbt.cli.main import cli; cli()",
           "build", "--project-dir", str(DBT_DIR), "--profiles-dir", str(DBT_DIR)]
    if subprocess.run(cmd, env=env).returncode != 0:
        raise SystemExit("dbt build failed")

Appending to run.py


**`publish()`**: the three models go to Iceberg. Created on the first run only; after that, compared with what the
table holds. Same rows → no commit (the re-run leaves the table untouched). Different rows → `overwrite()` **in
place**: a new snapshot, and the old one kept for time travel. Never drop and recreate.

In [7]:
%%writefile -a run.py


def publish(db: Path, warehouse: Path) -> None:
    """Step 5: publish three dbt models to Iceberg (see the brief for the exact columns).

        silver.product_cost_history   gold.weekly_margin   quality.price_quarantine

    Update existing tables in place, never drop and recreate them: the table's snapshots are
    how we see what a number was before the supplier's correction arrived.
    """
    catalog = get_catalog(warehouse)
    with duckdb.connect(str(db), read_only=True) as con:
        for name, keys in PUBLISH.items():
            data = con.sql(f"SELECT * FROM {name} ORDER BY {', '.join(keys)}").to_arrow_table()
            catalog.create_namespace_if_not_exists(name.split(".")[0])

            # First run: create the table. This is the only time a table is ever created.
            if not catalog.table_exists(name):
                table = catalog.create_table(name, schema=data.schema)
                table.append(data)
                print(f"publish: {name:30s} created, {data.num_rows:,} rows")
                continue

            # Later runs: compare with what the table holds now. Same rows -> no commit at all, so a re-run on the
            # same data leaves the table (and its snapshot list) exactly as it was.
            table = catalog.load_table(name)
            current = table.scan().to_arrow().select(data.column_names)
            if current.sort_by([(k, "ascending") for k in keys]).to_pylist() == data.to_pylist():
                print(f"publish: {name:30s} unchanged, {data.num_rows:,} rows ({len(table.snapshots())} snapshots)")
                continue

            # Something changed (e.g. the supplier's correction arrived): overwrite IN PLACE. Iceberg writes a new
            # snapshot and keeps the old one, so time travel still shows the numbers from before the change.
            table.overwrite(data)
            print(f"publish: {name:30s} updated, {data.num_rows:,} rows ({len(table.snapshots())} snapshots)")

Appending to run.py


**`main()`**: unchanged from the template.

In [8]:
%%writefile -a run.py






def main() -> None:
    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    ap.add_argument("--data-dir", type=Path, default=Path("data"))
    ap.add_argument("--warehouse", type=Path, default=Path("warehouse"))
    args = ap.parse_args()

    args.warehouse.mkdir(parents=True, exist_ok=True)
    db = args.warehouse / "qafila.duckdb"
    ingest(args.data_dir, db)
    transform(db)
    publish(db, args.warehouse)
    print(f"done: {args.warehouse.resolve()}")


if __name__ == "__main__":
    main()

Appending to run.py


Run step 1 alone and look at what landed:

In [9]:
import importlib

import run
importlib.reload(run)                                   # pick up the file we just wrote



DATA, WH = Path("data"), Path("warehouse")
WH.mkdir(exist_ok=True)
DB = WH / "qafila.duckdb"
run.ingest(DATA, DB)
with duckdb.connect(str(DB), read_only=True) as c:
    display(c.sql("""
      SELECT source_file, count(*) AS rows, count(unit_cost) AS unit_cost, count(cost_sar) AS cost_sar
      FROM raw.supplier_prices
      GROUP BY 1
      ORDER BY 1
    """).df())

ingest: orders 45,766, order_items 172,545, products 356, stores 12, supplier_prices 640 (4 supplier files)


,source_file,rows,unit_cost,cost_sar
0,prices_2026-W36.csv,160,160,0
1,prices_2026-W37.csv,160,160,0
2,prices_2026-W38.csv,160,160,0
3,prices_2026-W39.csv,160,0,160


## 2 · Clean: dbt staging models

The dbt project config. One addition to the template: the first supplier week as a **var**, the only date the brief
lets us fix. Everything else (file names, weeks, SKUs) comes from the data.

In [10]:
%%writefile dbt/dbt_project.yml
name: qafila_margin
version: "1.0.0"
profile: qafila_margin

model-paths: ["models"]
macro-paths: ["macros"]
test-paths: ["tests"]




vars:
  # The first supplier week. The only date the brief allows us to fix: everything else comes from the data.
  margin_start_date: "2026-08-31"

models:
  qafila_margin:
    staging:
      +schema: staging
      +materialized: view
    silver:
      +schema: silver
      +materialized: table
    gold:
      +schema: gold
      +materialized: table
    quality:
      +schema: quality
      +materialized: table

Overwriting dbt/dbt_project.yml


In [11]:
%%writefile dbt/models/staging/_sources.yml
version: 2

# Step 1 (run.py: ingest) lands these tables, exactly as they arrive (full refresh every run).
sources:
  - name: raw
    schema: raw
    tables:
      - name: orders
        description: App database orders. created_at / updated_at are UTC timestamps stored as text.
      - name: order_items
        description: App database order lines. A handful have a quantity of zero or less.
      - name: products
        description: App database products. Two SKUs have trailing spaces.
      - name: stores
        description: App database dark stores (12), each in one city.
      - name: supplier_prices
        description: >
          Every row of every suppliers/prices_*.csv file, all columns as text, with the file it came from
          (source_file). The cost is in unit_cost (up to W38) or cost_sar (W39 on).
        columns:
          - name: source_file
            data_tests:
              - not_null









Overwriting dbt/models/staging/_sources.yml


One staging model per source: types, trimming, dates. Staging cleans, it never filters.

In [12]:
%%writefile dbt/models/staging/stg_orders.sql
-- One row per order. The app stores UTC as text; Qafila counts days in Riyadh time (UTC+3),
-- so an order at 22:30 UTC on 30 August is a 31 August order.
select
    order_id,
    store_id,
    status,
    created_at::timestamp                                   as created_at_utc,
    (created_at::timestamp + interval 3 hour)::date         as order_date
from {{ source('raw', 'orders') }}

Writing dbt/models/staging/stg_orders.sql


In [13]:
%%writefile dbt/models/staging/stg_order_items.sql
-- One row per order line (order_id, line_no), with its revenue (rule 8).
-- Lines with quantity <= 0 are kept here: staging cleans, it doesn't decide. The silver model filters them.
select
    order_id,
    line_no,
    product_id,
    quantity,
    unit_price,
    discount,
    round(quantity * unit_price - discount, 2)              as revenue_sar
from {{ source('raw', 'order_items') }}

Writing dbt/models/staging/stg_order_items.sql


In [14]:
%%writefile dbt/models/staging/stg_products.sql
-- One row per product. SKUs are trimmed (rule 2): two of them carry trailing spaces in the app database,
-- and the supplier files write them without, so an untrimmed join would never find their cost.
select
    product_id,
    trim(sku)                                               as sku,
    name_en,
    category
from {{ source('raw', 'products') }}

Writing dbt/models/staging/stg_products.sql


In [15]:
%%writefile dbt/models/staging/stg_stores.sql
-- One row per dark store. The city of an order line comes from here (rule 10), not from the customer.
select
    store_id,
    city
from {{ source('raw', 'stores') }}

Writing dbt/models/staging/stg_stores.sql


**The supplier costs** (rules 1 and 2). `translate` turns Arabic-Indic digits into 0-9 and both decimal separators
(`٫` and `,`) into a dot; `try_cast` gives `NULL` instead of failing on anything that still isn't a number.

In [16]:
%%writefile dbt/models/staging/stg_supplier_prices.sql
-- One row per row of every supplier file: the cost parsed into a number, nothing filtered yet.
-- Deciding which rows are usable is the job of silver (history) and quality (quarantine).
with src as (
    select
        source_file,
        sku,
        valid_from,
        -- rule 1: the cost is in unit_cost, or in cost_sar from week 39 on (a file only ever has one of them)
        coalesce(cost_sar, unit_cost)                       as raw_cost
    from {{ source('raw', 'supplier_prices') }}
)

select
    source_file,
    trim(sku)                                               as sku,         -- rule 2: compare trimmed SKUs
    try_cast(trim(valid_from) as date)                      as valid_from,
    raw_cost,                                                               -- kept exactly as written, for the quarantine
    -- rule 1: '9,49' and '٩٫٤٩' both mean 9.49. Arabic-Indic digits become 0-9, and both the Arabic decimal
    -- separator (٫) and the decimal comma become a dot. try_cast gives NULL instead of failing on anything
    -- that still isn't a number, and NULL is quarantined as an invalid cost. Rounded to 2 decimals (rule 6).
    round(try_cast(translate(trim(raw_cost), '٠١٢٣٤٥٦٧٨٩٫,', '0123456789..') as double), 2) as unit_cost_sar
from src





Writing dbt/models/staging/stg_supplier_prices.sql


Staging tests watch the **inputs**, so they warn instead of failing: a surprise in an unseen dataset is reported, not fatal.

In [17]:
%%writefile dbt/models/staging/_staging.yml
version: 2

# Staging: one model per source, cleaning only (types, trimming, Riyadh dates, parsed costs). Views.
# Tests here watch the INPUTS, so they warn instead of failing: a surprise in an unseen dataset should
# be reported, not stop the run. The tests on the outputs (silver, gold, quality) are the strict ones.
models:
  - name: stg_orders
    description: One row per order, with its Riyadh order date.
    columns:
      - name: order_id
        data_tests:
          - unique
          - not_null
      - name: order_date
        description: created_at (UTC) + 3 hours, as a date. The date every rule uses.
        data_tests:
          - not_null:
              config:
                severity: warn

  - name: stg_order_items
    description: One row per order line, with revenue_sar = round(quantity * unit_price - discount, 2).
    data_tests:
      - unique:
          arguments:
            column_name: "order_id || '-' || line_no"

  - name: stg_products
    description: One row per product, SKU trimmed.
    columns:
      - name: sku
        description: Trimmed. Must stay unique after trimming, or a line would match two costs.
        data_tests:
          - unique:
              config:
                severity: warn
          - not_null:
              config:
                severity: warn

  - name: stg_stores
    description: One row per dark store.

  - name: stg_supplier_prices
    description: One row per supplier file row, cost parsed (decimal commas, Arabic-Indic digits), nothing filtered.
    columns:
      - name: valid_from
        data_tests:
          - not_null:
              config:
                severity: warn

Writing dbt/models/staging/_staging.yml


## 3 · History: quarantine and SCD2 cost history

**The quarantine rule lives in one model.** `supplier_price_checks` gives every supplier row a `reject_reason`
(`NULL` = usable). The history reads the usable rows, the quarantine reads the others: they can never disagree.

In [18]:


%%writefile dbt/models/silver/supplier_price_checks.sql
-- One row per row of every supplier file, with the reason it can't be used (NULL = usable).
-- The quarantine rule (rule 3) lives here and only here: product_cost_history reads the usable rows,
-- price_quarantine reads the others, so the two can never disagree.
select
    s.source_file,
    s.sku,
    s.valid_from,
    s.raw_cost,
    s.unit_cost_sar,
    case
        when p.sku is null                                      then 'unknown_sku'   -- checked first: an unknown SKU
        when s.unit_cost_sar is null or s.unit_cost_sar <= 0    then 'invalid_cost'  -- is rejected whatever its cost
    end                                                     as reject_reason
from {{ ref('stg_supplier_prices') }} s
left join {{ ref('stg_products') }} p
  on p.sku = s.sku                                          -- both sides already trimmed (rule 2)

Writing dbt/models/silver/supplier_price_checks.sql


In [19]:
%%writefile dbt/models/quality/price_quarantine.sql
-- One row per rejected supplier row, with the cost exactly as it appeared in the file.
-- This is the list that goes back to the supplier. The rule itself is in silver.supplier_price_checks.
select
    source_file,
    sku,
    valid_from,
    raw_cost,
    reject_reason                                           as reason      -- 'unknown_sku' or 'invalid_cost'
from {{ ref('supplier_price_checks') }}
where reject_reason is not null

Writing dbt/models/quality/price_quarantine.sql


**The SCD2 history**, in four CTEs, one rule each:
1. `usable`: quarantined rows are ignored (rule 3);
2. `quotes`: one quote per SKU and `valid_from`; the file name that sorts last wins (rule 5);
3. `changes`: each quote next to the SKU's previous one (`lag`). A week where a SKU isn't quoted has no row, so the
   old cost carries on by itself (rule 4);
4. `versions`: only a **different** cost starts a version (rule 6); `valid_to` = the next version's `valid_from` (`lead`).

In [20]:
%%writefile dbt/models/silver/product_cost_history.sql
-- SCD2 history of supplier costs: one row per SKU per cost version.
-- valid_from is inclusive, valid_to exclusive (NULL for the current version).
with usable as (
    -- rule 3: quarantined rows are ignored everywhere else
    select sku, valid_from, unit_cost_sar, source_file
    from {{ ref('supplier_price_checks') }}
    where reject_reason is null
),


quotes as (
    -- rule 5: the same SKU and valid_from in several files -> the file whose name sorts last wins.
    -- That's how prices_2026-W39_fix.csv replaces prices_2026-W39.csv, without naming either file.
    select sku, valid_from, unit_cost_sar, source_file
    from usable
    qualify row_number() over (partition by sku, valid_from order by source_file desc) = 1
),

changes as (
    -- rule 6: compare every quote with the SKU's previous quote. A week where the SKU isn't quoted has
    -- no row at all, so the previous cost simply carries on (rule 4) and the gap never starts a version.
    select
        sku,
        valid_from,
        unit_cost_sar,
        source_file,
        lag(unit_cost_sar) over (partition by sku order by valid_from) as previous_cost
    from quotes
),

versions as (
    -- a new version only when the cost changes: the first quote of a SKU, or a different cost
    select sku, valid_from, unit_cost_sar, source_file
    from changes
    where previous_cost is null
       or unit_cost_sar <> previous_cost
)

select
    sku,
    unit_cost_sar,
    valid_from,
    lead(valid_from) over w                                 as valid_to,      -- where the next version starts
    lead(valid_from) over w is null                         as is_current,
    source_file                                                               -- lineage: the file that set this cost
from versions
window w as (partition by sku order by valid_from)

Writing dbt/models/silver/product_cost_history.sql


## 4 · Margin: point-in-time costing and the weekly table

The grain of the margin: **one row per counted order line**. The point-in-time join finds the version where
`valid_from <= order_date < valid_to`. It's a `left join`: a line without a valid cost stays in, uncosted.

In [21]:
%%writefile dbt/models/silver/order_lines_costed.sql
-- One row per counted order line (rule 7), with the supplier cost valid on its order date.
-- The grain of the margin: gold.weekly_margin only adds these rows up.
with lines as (
    select
        i.order_id,
        i.line_no,
        o.order_date,
        s.city,                                             -- rule 10: city from the order's store
        p.category,                                         -- ...and category from the product
        p.sku,
        i.quantity,
        i.revenue_sar                                       -- rule 8, computed in staging
    from {{ ref('stg_order_items') }} i
    join {{ ref('stg_orders') }} o
      on o.order_id = i.order_id
    -- left joins: a line whose store or product is missing still counts in lines and revenue
    left join {{ ref('stg_stores') }} s
      on s.store_id = o.store_id
    left join {{ ref('stg_products') }} p
      on p.product_id = i.product_id
    where o.status = 'delivered'
      and i.quantity > 0
      and o.order_date >= date '{{ var("margin_start_date") }}'
)

select
    l.*,
    h.unit_cost_sar,
    h.valid_from                                            as cost_valid_from,
    l.quantity * h.unit_cost_sar                            as cost_sar,      -- rule 9 (NULL when uncosted)
    h.sku is not null                                       as is_costed
from lines l
-- rule 9, the point-in-time join: the version where valid_from <= order date < valid_to.
-- Versions of a SKU never overlap, so a line matches at most one; a line with no match (SKU never quoted,
-- or ordered before its first quote) stays in with a NULL cost: an uncosted line.
left join {{ ref('product_cost_history') }} h
  on h.sku = l.sku
 and l.order_date >= h.valid_from
 and (l.order_date < h.valid_to or h.valid_to is null)

Writing dbt/models/silver/order_lines_costed.sql


The gold table only adds those lines up. The margin is computed from the rounded totals, so it adds up to the halala.

In [22]:
%%writefile dbt/models/gold/weekly_margin.sql
-- Gross margin: one row per ISO week x city x category that has at least one counted order line.
-- Uncosted lines count in lines and revenue_sar, but not in the cost columns (rule 9).
with weekly as (
    select
        weekofyear(order_date)::int                                         as iso_week,   -- ISO week (rule 10)
        city,
        category,
        count(*)::int                                                       as lines,
        round(sum(revenue_sar), 2)                                          as revenue_sar,
        round(coalesce(sum(revenue_sar) filter (where is_costed), 0), 2)    as costed_revenue_sar,
        round(coalesce(sum(cost_sar), 0), 2)                                as cost_sar,
        (count(*) filter (where not is_costed))::int                        as uncosted_lines
    from {{ ref('order_lines_costed') }}
    group by all
)

-- the margin from the rounded totals, so margin_sar = costed_revenue_sar - cost_sar holds to the halala
select
    iso_week,
    city,
    category,
    lines,
    revenue_sar,
    costed_revenue_sar,
    cost_sar,
    round(costed_revenue_sar - cost_sar, 2)                                 as margin_sar,
    uncosted_lines
from weekly

Writing dbt/models/gold/weekly_margin.sql


## 5 · Prove it: tests, a run, a re-run

Generic tests in YAML (grain, not null, accepted values)...

In [23]:
%%writefile dbt/models/silver/_silver.yml
version: 2

models:
  - name: supplier_price_checks
    description: >
      One row per supplier file row with reject_reason (NULL = usable). The only place the quarantine rule
      (rule 3) is written: unknown_sku first, then invalid_cost.
    columns:
      - name: reject_reason
        data_tests:
          - accepted_values:
              arguments:
                values: ["unknown_sku", "invalid_cost"]

  - name: product_cost_history
    description: >
      SCD2 history of supplier costs. Grain: one row per SKU per cost version. valid_from is inclusive,
      valid_to exclusive and NULL for the current version. A version starts only when the cost changes.
    data_tests:
      - unique:
          arguments:
            column_name: "sku || '|' || valid_from"
    columns:
      - name: sku
        data_tests:
          - not_null
      - name: valid_from
        data_tests:
          - not_null
      - name: unit_cost_sar
        data_tests:
          - not_null

  - name: order_lines_costed
    description: >
      Grain: one row per counted order line (delivered, quantity > 0, order date on or after the first
      supplier week), with the cost version valid on its order date. Uncosted lines have a NULL cost.
    data_tests:
      - unique:
          # the point-in-time join must never match two versions: that would count a line twice
          arguments:
            column_name: "order_id || '-' || line_no"
    columns:
      - name: revenue_sar
        data_tests:
          - not_null

Writing dbt/models/silver/_silver.yml


In [24]:
%%writefile dbt/models/gold/_gold.yml
version: 2

models:
  - name: weekly_margin
    description: >
      Gross margin per week. Grain: one row per iso_week x city x category with at least one counted line.
      margin_sar = costed_revenue_sar - cost_sar; uncosted lines count in lines and revenue_sar only.
    data_tests:
      - unique:
          arguments:
            column_name: "iso_week || '|' || city || '|' || category"
    columns:
      - name: iso_week
        data_tests:
          - not_null
      - name: lines
        data_tests:
          - not_null

Writing dbt/models/gold/_gold.yml


In [25]:
%%writefile dbt/models/quality/_quality.yml
version: 2

models:
  - name: price_quarantine
    description: >
      One row per rejected supplier row: unknown_sku (not sold by Qafila) or invalid_cost (zero, negative or
      not a number). raw_cost is the value exactly as it appeared in the file. Grain: one row per supplier file row.
    columns:
      - name: source_file
        data_tests:
          - not_null
      - name: reason
        data_tests:
          - not_null
          - accepted_values:
              arguments:
                values: ["unknown_sku", "invalid_cost"]

Writing dbt/models/quality/_quality.yml


...and singular tests: plain SQL in `tests/` that must return **no rows**.

In [26]:
%%writefile dbt/tests/assert_every_supplier_row_is_used_or_quarantined.sql
-- No supplier row may disappear: every row of every file is either usable or in the quarantine.
select *
from (
  select
    (select count(*) from {{ ref('stg_supplier_prices') }})                                       as file_rows,
    (select count(*) from {{ ref('supplier_price_checks') }} where reject_reason is null)         as usable_rows,
    (select count(*) from {{ ref('price_quarantine') }})                                          as quarantined_rows
)
where file_rows <> usable_rows + quarantined_rows

Writing dbt/tests/assert_every_supplier_row_is_used_or_quarantined.sql

In [27]:
%%writefile dbt/tests/assert_history_costs_are_positive.sql
-- No zero or negative cost may reach the history: those rows belong in the quarantine (rule 3).
select *
from {{ ref('product_cost_history') }}
where unit_cost_sar <= 0

Writing dbt/tests/assert_history_costs_are_positive.sql

In [28]:
%%writefile dbt/tests/assert_history_one_current_version_per_sku.sql
-- Every SKU in the history has exactly one current version (the one with an empty valid_to).
-- Returns the SKUs that break it: the test passes when this returns nothing.
select
  sku,
  count(*) filter (where is_current) as current_versions
from {{ ref('product_cost_history') }}
group by sku
having count(*) filter (where is_current) <> 1
    or count(*) filter (where is_current and valid_to is not null) > 0

Writing dbt/tests/assert_history_one_current_version_per_sku.sql


In [29]:
%%writefile dbt/tests/assert_history_versions_chain_without_gaps.sql
-- SCD2 chain: each version ends exactly where the SKU's next version starts (no gap, no overlap),
-- every version is at least one day long, and the cost really changes between two versions (rule 6).
with chained as (
  select
    sku,
    valid_from,
    valid_to,
    unit_cost_sar,
    lead(valid_from) over (partition by sku order by valid_from)    as next_valid_from,
    lag(unit_cost_sar) over (partition by sku order by valid_from)  as previous_cost
  from {{ ref('product_cost_history') }}
)

select *
from chained
where valid_to is distinct from next_valid_from      -- a gap or an overlap
   or valid_to <= valid_from                         -- an empty or backwards version
   or unit_cost_sar = previous_cost                  -- a "new" version with the same cost

Writing dbt/tests/assert_history_versions_chain_without_gaps.sql


In [30]:
%%writefile dbt/tests/assert_margin_adds_up.sql
-- Internal consistency of every margin row: the margin is costed revenue minus cost, costed revenue is
-- part of the revenue, and uncosted lines are part of the lines.
select *
from {{ ref('weekly_margin') }}
where abs(costed_revenue_sar - cost_sar - margin_sar) > 0.01
   or costed_revenue_sar > revenue_sar + 0.01
   or uncosted_lines > lines
   or uncosted_lines < 0

Writing dbt/tests/assert_margin_adds_up.sql


In [31]:
%%writefile dbt/tests/assert_margin_reconciles_with_source.sql
-- Reconciliation: the margin table must hold every eligible order line and all its revenue, counted
-- straight from staging. If a join silently dropped or duplicated lines, the two sides differ.
with source_lines as (
  select
    count(*)                  as lines,
    round(sum(i.revenue_sar), 2) as revenue_sar
  from {{ ref('stg_order_items') }} i
  join {{ ref('stg_orders') }} o
    on o.order_id = i.order_id
  where o.status = 'delivered'
    and i.quantity > 0
    and o.order_date >= date '{{ var("margin_start_date") }}'
),

margin as (
  select
    sum(lines)               as lines,
    round(sum(revenue_sar), 2) as revenue_sar
  from {{ ref('weekly_margin') }}
)

select *
from source_lines s, margin m
where s.lines <> m.lines
   or abs(s.revenue_sar - m.revenue_sar) > 1    -- per-row rounding may move the total by a few halalas

Writing dbt/tests/assert_margin_reconciles_with_source.sql


The whole pipeline, on an empty warehouse: `dbt build` runs every model and its tests in dependency order.

In [32]:
out = py("run.py")
print("\n".join(l for l in out.splitlines() if "Done." in l or l.startswith(("ingest", "publish", "done"))))

05:54:22  Done. PASS=37 WARN=0 ERROR=0 SKIP=0 NO-OP=0 REUSED=0 TOTAL=37
ingest: orders 45,766, order_items 172,545, products 356, stores 12, supplier_prices 640 (4 supplier files)
publish: silver.product_cost_history    created, 633 rows
publish: gold.weekly_margin             created, 165 rows
publish: quality.price_quarantine       created, 7 rows
done: C:\Users\vmuil\qafila-nbtest\warehouse


Look at one SKU's history: a version per cost change, `valid_to` where the next one starts.

In [33]:
with duckdb.connect(str(DB), read_only=True) as c:
    display(c.sql("""
      SELECT *
      FROM silver.product_cost_history
      WHERE sku = (SELECT sku FROM silver.product_cost_history GROUP BY sku ORDER BY count(*) DESC, sku LIMIT 1)
      ORDER BY valid_from
    """).df())
    display(c.sql("SELECT * FROM quality.price_quarantine ORDER BY source_file, sku").df())
    display(c.sql("SELECT * FROM gold.weekly_margin ORDER BY iso_week, city, category LIMIT 8").df())

,sku,unit_cost_sar,valid_from,valid_to,is_current,source_file
0,QF-BAK-1066,4.08,2026-08-31,2026-09-07,False,prices_2026-W36.csv
1,QF-BAK-1066,3.34,2026-09-07,2026-09-14,False,prices_2026-W37.csv
2,QF-BAK-1066,3.28,2026-09-14,2026-09-21,False,prices_2026-W38.csv
3,QF-BAK-1066,3.12,2026-09-21,NaT,True,prices_2026-W39.csv


,source_file,sku,valid_from,raw_cost,reason
0,prices_2026-W39.csv,QF-DAI-1006,2026-09-21,-1,invalid_cost
1,prices_2026-W39.csv,QF-DAI-1051,2026-09-21,-1,invalid_cost
2,prices_2026-W39.csv,QF-DAT-1201,2026-09-21,0,invalid_cost
3,prices_2026-W39.csv,QF-FRU-1086,2026-09-21,-5.5,invalid_cost
4,prices_2026-W39.csv,QF-NEW-9292,2026-09-21,18.34,unknown_sku
5,prices_2026-W39.csv,QF-NEW-9413,2026-09-21,6.92,unknown_sku
6,prices_2026-W39.csv,QF-NEW-9435,2026-09-21,19.56,unknown_sku


,iso_week,city,category,lines,revenue_sar,costed_revenue_sar,cost_sar,margin_sar,uncosted_lines
0,36,Dammam,Bakery,179,1739.23,791.27,544.94,246.33,110
1,36,Dammam,Beverages,297,9027.04,5721.60,3708.99,2012.61,103
2,36,Dammam,Dairy,497,15767.01,2827.13,1866.31,960.82,352
3,36,Dammam,Dates & Coffee,307,35102.61,9732.26,6291.53,3440.73,223
4,36,Dammam,Frozen,173,4759.58,1589.56,1185.67,403.89,134
5,36,Dammam,Fruits & Vegetables,556,10184.49,4111.80,2823.06,1288.74,355
6,36,Dammam,Household,144,6241.51,4535.90,3079.39,1456.51,53
7,36,Dammam,Meat & Poultry,290,25933.23,15916.28,10303.44,5612.84,86


**Re-run on the same data**: `publish()` should find nothing to change.

In [34]:
print("\n".join(l for l in py("run.py").splitlines() if l.startswith("publish")))

publish: silver.product_cost_history    unchanged, 633 rows (1 snapshots)
publish: gold.weekly_margin             unchanged, 165 rows (1 snapshots)
publish: quality.price_quarantine       unchanged, 7 rows (1 snapshots)


The contract self-check that ships with the project:

In [35]:
print(py("tools/check.py"))

✅ Iceberg catalog at C:\Users\vmuil\qafila-nbtest\warehouse\catalog.db
✅ table silver.product_cost_history: 633 rows, columns as in the brief
✅ table gold.weekly_margin: 165 rows, columns as in the brief
✅ table quality.price_quarantine: 7 rows, columns as in the brief
✅ history: one row per (sku, valid_from)
✅ history: exactly one current version per SKU
✅ history: valid_to is empty exactly when is_current
✅ history: each version ends where the next one starts (valid_to is exclusive)
✅ history: every cost is positive
✅ history: a new version only when the cost changes
✅ quarantine: reason is 'unknown_sku' or 'invalid_cost'
✅ quarantine: the planted bad supplier rows are there
✅ margin: one row per (iso_week, city, category)
✅ margin: iso_week between 36 and 40
✅ margin: margin_sar = costed_revenue_sar - cost_sar
✅ margin: costed_revenue_sar <= revenue_sar
✅ margin: uncosted_lines <= lines
✅ margin: lines and revenue cover every eligible order line
✅ Iceberg: gold.weekly_margin has 1 s

## 6 · The correction arrives: update in place, time travel

The supplier sends `prices_2026-W39_fix.csv`. Nothing in the code names it: rule 5 (the later file name wins) picks
it up. Run again, then read `gold.weekly_margin` **as of each snapshot**.

In [36]:
print(py("tools/make_data.py", "--add-correction"))
print("\n".join(l for l in py("run.py").splitlines() if l.startswith(("ingest", "publish"))))

wrote C:\Users\vmuil\qafila-nbtest\data\suppliers\prices_2026-W39_fix.csv



ingest: orders 45,766, order_items 172,545, products 356, stores 12, supplier_prices 649 (5 supplier files)
publish: silver.product_cost_history    updated, 637 rows (3 snapshots)
publish: gold.weekly_margin             updated, 165 rows (3 snapshots)
publish: quality.price_quarantine       unchanged, 7 rows (1 snapshots)


In [37]:
margin = run.get_catalog(WH).load_table("gold.weekly_margin")
rows = []
for s in margin.snapshots():
    df = margin.scan(snapshot_id=s.snapshot_id).to_pandas()
    rows.append({"snapshot_id": s.snapshot_id, "operation": s.summary.operation.value, "rows": len(df),
                 "cost_sar": round(df.cost_sar.sum(), 2), "margin_sar": round(df.margin_sar.sum(), 2)})
pd.DataFrame(rows)


,snapshot_id,operation,rows,cost_sar,margin_sar
0,7034804823456180946,append,165,1654199.98,801016.88
1,1931290804966521438,delete,0,0.00,0.00
2,8311135978169494855,append,165,1651701.06,803515.80


The first snapshot still holds the numbers from before the correction; the last one holds the corrected ones.
`overwrite` shows up as a `delete` + an `append`. `tools/evidence.py` prints these and the data-problem counts used in the README:

In [38]:
%%writefile tools/evidence.py
"""Print the evidence the README quotes: data problems (with counts) and the Iceberg history of the outputs.

    uv run python tools/evidence.py --data-dir data --warehouse warehouse

Read-only: it opens the DuckDB file read-only and only scans the Iceberg tables.
"""

from __future__ import annotations

import argparse
import sys
from pathlib import Path

import duckdb

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from run import get_catalog  # noqa: E402  (the same catalog the pipeline and the grader use)




def data_problems(data_dir: Path, db: Path) -> None:
    """Count every problem from DATA_ISSUES.txt that touches this project, straight from raw / staging."""
    with duckdb.connect(str(db), read_only=True) as con:
        q = lambda sql: con.sql(sql).fetchone()[0]                    # noqa: E731
        counts = {
            "decimal commas (9,49)": q(r"SELECT count(*) FROM raw.supplier_prices WHERE unit_cost LIKE '%,%'"),
            "Arabic-Indic digits (٩٫٤٩)": q("SELECT count(*) FROM raw.supplier_prices WHERE regexp_matches(coalesce(cost_sar, unit_cost), '[٠-٩]')"),
            "rows using cost_sar (renamed column)": q("SELECT count(*) FROM raw.supplier_prices WHERE cost_sar IS NOT NULL"),
            "zero / negative / unparseable cost": q("SELECT count(*) FROM quality.price_quarantine WHERE reason = 'invalid_cost'"),
            "SKUs Qafila doesn't sell": q("SELECT count(*) FROM quality.price_quarantine WHERE reason = 'unknown_sku'"),
            "product SKUs with trailing spaces": q("SELECT count(*) FROM raw.products WHERE sku <> trim(sku)"),
            "order lines with quantity <= 0": q("SELECT count(*) FROM raw.order_items WHERE quantity <= 0"),
            "rows replaced by a later file (rule 5)": q("""
                SELECT count(*) - count(DISTINCT (sku, valid_from))
                FROM silver.supplier_price_checks WHERE reject_reason IS NULL"""),
            "repeated costs merged into one version": q("""
                SELECT count(DISTINCT (sku, valid_from)) FROM silver.supplier_price_checks WHERE reject_reason IS NULL""")
                - q("SELECT count(*) FROM silver.product_cost_history"),
            "counted lines without a valid cost (uncosted)": q("SELECT sum(uncosted_lines) FROM gold.weekly_margin"),
        }
    boms = [f.name for f in sorted((data_dir / "suppliers").glob("prices_*.csv")) if f.read_bytes()[:3] == b"\xef\xbb\xbf"]
    print("Data problems")
    for name, n in counts.items():
        print(f"  {name:48s} {n:>6,}")
    print(f"  {'files starting with a byte-order mark':48s} {', '.join(boms) or 'none'}")


def iceberg_history(warehouse: Path) -> None:
    """Every snapshot of the three tables; for gold.weekly_margin, the totals each snapshot holds (time travel)."""
    catalog = get_catalog(warehouse)
    print("\nIceberg snapshots")
    for name in ["silver.product_cost_history", "gold.weekly_margin", "quality.price_quarantine"]:
        table = catalog.load_table(name)
        print(f"  {name}")
        for s in table.snapshots():
            rows = table.scan(snapshot_id=s.snapshot_id).to_arrow()
            line = f"    {s.snapshot_id:<20} {s.summary.operation.value:9s} rows {rows.num_rows:>4}"
            if name == "gold.weekly_margin" and rows.num_rows:
                df = rows.to_pandas()
                line += f" | cost_sar {df.cost_sar.sum():>12,.2f} | margin_sar {df.margin_sar.sum():>12,.2f}"
            print(line)


def main() -> None:
    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    ap.add_argument("--data-dir", type=Path, default=Path("data"))
    ap.add_argument("--warehouse", type=Path, default=Path("warehouse"))
    args = ap.parse_args()
    data_problems(args.data_dir, args.warehouse / "qafila.duckdb")
    iceberg_history(args.warehouse)


if __name__ == "__main__":
    main()




Writing tools/evidence.py


In [39]:
print(py("tools/evidence.py"))


Data problems
  decimal commas (9,49)                               126
  Arabic-Indic digits (٩٫٤٩)                           34
  rows using cost_sar (renamed column)                169
  zero / negative / unparseable cost                    4
  SKUs Qafila doesn't sell                              3
  product SKUs with trailing spaces                     2
  order lines with quantity <= 0                      142
  rows replaced by a later file (rule 5)                5
  repeated costs merged into one version                0
  counted lines without a valid cost (uncosted)    20,626
  files starting with a byte-order mark            prices_2026-W38.csv

Iceberg snapshots
  silver.product_cost_history
    6934996885322553685  append    rows  633
    4996640233549468394  delete    rows    0
    3879039162421507950  append    rows  637
  gold.weekly_margin
    7034804823456180946  append    rows  165 | cost_sar 1,654,199.98 | margin_sar   801,016.88
    1931290804966521438  delete    